[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/11_dp.ipynb)

*DSA Interview Handbook · Notebook 11 of 14*

# Dynamic Programming

Recursion that remembers: define a state, write the recurrence, fill a table in the right order.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Dynamic programming is recursion with a notebook: never solve the same small problem twice.

**What interviewers really test here:** Defining the state and the transition, and moving from memo to table.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why plain recursion is slow *(Basics)*
- [ ] 2. Two ways to do DP *(Basics)*
- [ ] 3. 1-D DP: take it or skip it *(Core)*
- [ ] 4. Knapsack: choosing items under a limit *(Core)*
- [ ] 5. Grid DP *(Core)*
- [ ] 6. Two-string DP *(Core)*
- [ ] 7. Harder DP shapes *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

**Dynamic programming (DP)** is an algorithm design technique for problems that can be broken into overlapping sub-problems. It solves each sub-problem **once**, stores the result, and reuses it — trading memory for time. The term was coined by Richard Bellman in the 1950s.

### When DP applies: the two properties

#### 1. Optimal substructure



An optimal solution to the problem contains optimal solutions to its sub-problems. Example: the shortest path A → C through B consists of the shortest path A → B and the shortest path B → C.



#### 2. Overlapping sub-problems



The same sub-problems are solved many times by a naive recursion. Example: `fib(n)` calls `fib(n − 2)` both directly and through `fib(n − 1)`.



If sub-problems **don't overlap** (merge sort's halves are independent), the right technique is **divide and conquer**, not DP. If a **local choice** is always safe, **greedy** is simpler.

### Approaches

|   | Top-down (memoisation) | Bottom-up (tabulation) |
| --- | --- | --- |
| Idea | recursion + cache of computed results | iteratively fill a table from base cases |
| Order | computed on demand | must choose an order where dependencies come first |
| Visits | only states actually needed | usually every state |
| Overhead | function calls, recursion depth limit | none; tight loops |
| Space optimisation | harder | easy (keep only the rows you need) |
| Python | `@functools.cache` | `dp = [0] * (n + 1)` |

### Designing a DP solution

1. **Define the state**: the smallest set of parameters that describes a sub-problem, e.g. `dp[i]` = best answer for the first i items; `dp[i][w]` = best value using items 0..i with capacity w.
2. **Write the recurrence (transition)**: how a state is computed from smaller states.
3. **Identify base cases**.
4. **Decide the evaluation order** (bottom-up) or add memoisation (top-down).
5. **Locate the answer** in the table (`dp[n]`, `max(dp)`, `dp[m][n]`).
6. **Optimise space** if each state depends only on the previous row/values.



**Complexity** = (number of states) × (work per transition). Example: LCS has m·n states with O(1) work → O(m·n).

### Classic DP problem families

| Family | State | Examples | Typical time |
| --- | --- | --- | --- |
| Linear / 1-D | dp[i] | Fibonacci, climbing stairs, house robber, decode ways | O(n) |
| 0/1 knapsack | dp[i][w] or dp[w] | subset sum, partition, target sum | O(n · W) |
| Unbounded knapsack | dp[w] | coin change, rod cutting | O(n · W) |
| Subsequences | dp[i] / tails | LIS | O(n²) or O(n log n) |
| Two sequences | dp[i][j] | LCS, edit distance, interleaving strings | O(m · n) |
| Grid | dp[r][c] | unique paths, min path sum, maximal square | O(m · n) |
| Interval | dp[l][r] | matrix chain multiplication, burst balloons, palindromic substrings | O(n³) / O(n²) |
| State machine | dp[i][state] | stock buy/sell with cooldown or fee | O(n) |
| Bitmask | dp[mask] | travelling salesman, assignment | O(2ⁿ · n²) |
| Tree DP | dp[node] | house robber III, tree diameter | O(n) |



**Pseudo-polynomial**: knapsack's O(n · W) depends on the *value* W, not just the input size, so it's slow for huge capacities.

### DP vs other paradigms

| Paradigm | Sub-problems | Choice | Example |
| --- | --- | --- | --- |
| Divide and conquer | independent | — | merge sort, quicksort |
| **Dynamic programming** | overlapping | tries all choices, keeps the best | knapsack, LCS |
| Greedy | one at a time | commits to the locally best choice | interval scheduling, Huffman coding |
| Backtracking | explores a tree of choices | prunes invalid branches | N-Queens, Sudoku |

### Applications

- Text: spell checkers and diff tools (edit distance, LCS); DNA sequence alignment in bioinformatics.
- Resource allocation and budgeting (knapsack variants).
- Shortest paths: Bellman–Ford and Floyd–Warshall are DP algorithms.
- Speech recognition and part-of-speech tagging (Viterbi algorithm).
- Finance: option pricing, optimal trading strategies.
- Reinforcement learning: value iteration, Bellman equations.

---
# Part 2: Lessons, step by step

## Lesson 1: Why plain recursion is slow

*Level: Basics*

Fibonacci: `fib(n) = fib(n − 1) + fib(n − 2)`. Written as plain recursion, `fib(5)` calls `fib(3)` twice, `fib(2)` three times, and so on. The number of calls roughly doubles with n — **O(2ⁿ)**. `fib(50)` would take hours.



```text
               fib(5)
          /            \
      fib(4)          fib(3)      ← fib(3) computed again
     /     \          /    \
  fib(3)  fib(2)   fib(2) fib(1)
```



The same small questions are asked again and again — **overlapping sub-problems**. That's the signal for dynamic programming.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def calls_lesson():
    calls = {}

    def fib(n):
        calls[n] = calls.get(n, 0) + 1
        if n <= 1:
            return n
        return fib(n - 1) + fib(n - 2)

    fib(6)
    memo, mcalls = {}, {}

    def fib_memo(n):
        mcalls[n] = mcalls.get(n, 0) + 1
        if n <= 1:
            return n
        if n not in memo:
            memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
        return memo[n]

    fib_memo(6)
    return fib_memo(6)


calls_lesson()

8

## Lesson 2: Two ways to do DP

*Level: Basics*

**Dynamic programming** = solve each sub-problem once and store the answer.



|   | Top-down (memoisation) | Bottom-up (tabulation) |
| --- | --- | --- |
| How | write the recursion, add a cache | fill a table from the smallest case up |
| Python | `@functools.cache` | `dp = [0] * (n + 1)` and a loop |
| Pros | easy to write from the recurrence; only visits needed states | no recursion limit; easy to save memory |



#### The four questions for every DP




1. **State**: what does `dp[i]` mean? Write it as a sentence.
2. **Transition**: how is `dp[i]` built from smaller states?
3. **Base cases**: the smallest states you know directly.
4. **Answer & order**: which cell is the answer, and in what order to fill so dependencies are ready.



Climbing stairs: state = ways to reach step i; transition = `dp[i − 1] + dp[i − 2]`; base `dp[0] = dp[1] = 1`; answer `dp[n]`.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2). Try one now, or carry on and do them in Part 3.

## Lesson 3: 1-D DP: take it or skip it

*Level: Core*

A huge family of problems walks a sequence and, at each element, makes a choice.




- **House robber**: `dp[i] = max(dp[i − 1], dp[i − 2] + nums[i])` — skip this house, or rob it and the best from two back.
- **Decode ways**: `dp[i] = dp[i − 1]` (one digit) `+ dp[i − 2]` (two digits, if 10–26).
- **Word break**: `dp[i]` is true if some `dp[j]` is true and `s[j:i]` is a word.
- **Circular houses**: run the line version twice — without the first house and without the last.



**💡 Tip:** If `dp[i]` only uses the last one or two values, keep two variables instead of a list — O(1) memory.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def robber_lesson():
    nums = [2, 7, 9, 3, 1]
    dp = [0] * len(nums)
    for i, x in enumerate(nums):
        skip = dp[i - 1] if i >= 1 else 0
        take = (dp[i - 2] if i >= 2 else 0) + x
        dp[i] = max(skip, take)
    return dp[-1]


robber_lesson()

12

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. House 0: skip → 0, rob → 2; best 2
2. House 1: skip → 2, rob → 7; best 7
3. House 2: skip → 7, rob → 11; best 11
4. House 3: skip → 11, rob → 10; best 11
5. House 4: skip → 11, rob → 12; best 12

</details>


**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q4](#q4), [Q14](#q14), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 4: Knapsack: choosing items under a limit

*Level: Core*

**0/1 knapsack**: each item can be used **once**. State `dp[c]` = best (or "possible?", or "number of ways") using capacity c. For each item, loop capacity **downwards** so the item isn't used twice in the same round:



```python
for x in items:
    for c in range(cap, x - 1, -1):
        dp[c] = dp[c] or dp[c - x]       # partition equal subset sum
```



**Unbounded knapsack**: items reusable (coins). Loop capacity **upwards**.



```python
dp = [0] + [inf] * amount
for a in range(1, amount + 1):
    for c in coins:
        if c <= a: dp[a] = min(dp[a], dp[a - c] + 1)   # coin change
```



Counting **combinations** (coin change II): coins in the outer loop. Coins in the inner loop would count orderings (permutations) instead.



Target sum turns into knapsack with algebra: the "+" group must sum to `(total + target) / 2`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def knapsack_lesson():
    nums = [1, 5, 11, 5]
    target = sum(nums) // 2
    dp = [True] + [False] * target
    for x in nums:
        for s in range(target, x - 1, -1):
            dp[s] = dp[s] or dp[s - x]
    return dp[target]


knapsack_lesson()

True

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. dp[s] = can some subset sum to s? Only s = 0 at first. Target 11
2. After number 1 (looping s downwards so 1 is used once)
3. After number 5 (looping s downwards so 5 is used once)
4. After number 11 (looping s downwards so 11 is used once)
5. After number 5 (looping s downwards so 5 is used once)

</details>


**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q17](#q17), [Q5](#q5), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 5: Grid DP

*Level: Core*

On a grid where you move only right or down, cell `(r, c)` is reached from the top or the left:




- **Unique paths**: `dp[r][c] = dp[r − 1][c] + dp[r][c − 1]`.
- **Minimum path sum**: `dp[r][c] = grid[r][c] + min(top, left)`.
- **Maximal square**: largest square ending at (r, c) = `1 + min(top, left, top-left)` when the cell is 1.



The first row and first column have only one way in — fill them as base cases. Since each row only needs the row above, one 1-D array is enough.

**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q13](#q13), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 6: Two-string DP

*Level: Core*

For two strings `a` and `b`, use `dp[i][j]` = answer for the prefixes `a[:i]` and `b[:j]`. Look at the last characters:




- **Longest common subsequence**: equal → `dp[i−1][j−1] + 1`; else `max(dp[i−1][j], dp[i][j−1])`.
- **Edit distance**: equal → `dp[i−1][j−1]`; else `1 + min(delete, insert, replace)` = `1 + min(dp[i−1][j], dp[i][j−1], dp[i−1][j−1])`.



Row 0 and column 0 represent an empty prefix and are the base cases. O(m · n) time; the table can shrink to two rows.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def lcs_lesson():
    a, b = "ace", "abcde"
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]


lcs_lesson()

3

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. 'a' = 'a' → diagonal + 1 = 1
2. 'c' = 'c' → diagonal + 1 = 2
3. 'e' = 'e' → diagonal + 1 = 3
4. LCS length = 3

</details>


**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 7: Harder DP shapes

*Level: Advanced*

#### Longest increasing subsequence



O(n²): `dp[i] = 1 + max(dp[j])` over earlier smaller values. O(n log n): keep `tails[k]` = smallest tail of an increasing subsequence of length k + 1, and binary search where each new number goes.



#### State machines



When each day can be in a few modes (holding a stock, just sold, resting), keep one variable per mode and write how each mode is reached from yesterday's modes.



#### Max and min together



Maximum product subarray: a negative number flips the biggest into the smallest, so track both.



#### Interval DP



`dp[l][r]` for a range, filled by increasing length. Burst balloons: choose the balloon burst **last** inside (l, r) — then its neighbours are fixed as l and r. Palindromes: expand from centres, or `dp[l][r] = s[l] == s[r] and dp[l+1][r−1]`.



**💡 Tip:** Stuck on the state? Write the brute-force recursion first. Whatever its arguments are, that's your DP state.

**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q18](#q18), [Q16](#q16), [Q20](#q20), [Q15](#q15). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Climbing Stairs

🟢 Warm-up · *1-D · Fibonacci*

You climb 1 or 2 steps at a time. How many distinct ways reach the top of `n` steps?

**Examples**

- `n = 2` → `2`
- `n = 3` → `3`
- `n = 5` → `8`

In [ ]:
def climb_stairs(n):
    """
    You climb 1 or 2 steps at a time. How many distinct ways reach the top of n
    steps?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(climb_stairs, [
    ((2,), 2),
    ((3,), 3),
    ((5,), 8),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

ways(n) = ways(n − 1) + ways(n − 2) as plain recursion.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same recurrence bottom-up; only the last two values are needed.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** ways(n) = ways(n − 1) + ways(n − 2) as plain recursion.

```python
def climb_stairs(n):
    if n <= 2:
        return n
    return climb_stairs(n - 1) + climb_stairs(n - 2)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Same recurrence bottom-up; only the last two values are needed.

```python
def climb_stairs(n):
    a, b = 1, 1  # ways to reach step i - 1 and step i
    for i in range(2, n + 1):
        a, b = b, a + b
    return b
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: ways(n) = ways(n − 1) + ways(n − 2) as plain recursion. That costs O(2ⁿ) time and O(n) space."

"I can do better: Same recurrence bottom-up; only the last two values are needed. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 6`

1. ways(2) = ways(1) + ways(0) = 2
2. ways(3) = ways(2) + ways(1) = 3
3. ways(4) = ways(3) + ways(2) = 5
4. ways(5) = ways(4) + ways(3) = 8
5. ways(6) = ways(5) + ways(4) = 13

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/climbing-stairs)

<a id="q2"></a>
### Q2. Min Cost Climbing Stairs

🟢 Warm-up · *1-D · min of two*

`cost[i]` is paid when you step on stair i. Start at stair 0 or 1, climb 1 or 2 at a time, and return the minimum cost to pass the last stair.

**Examples**

- `cost = [10, 15, 20]` → `15`
- `cost = [1, 100, 1, 1, 1, 100, 1, 1, 100, 1]` → `6`

In [ ]:
def min_cost_climbing_stairs(cost):
    """
    cost[i] is paid when you step on stair i. Start at stair 0 or 1, climb 1 or
    2 at a time, and return the minimum cost to pass the last stair.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_cost_climbing_stairs, [
    (([10, 15, 20],), 15),
    (([1, 100, 1, 1, 1, 100, 1, 1, 100, 1],), 6),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursively try both moves from every stair.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

dp[i] = cost to reach stair i = min(dp[i − 1] + cost[i − 1], dp[i − 2] + cost[i − 2]).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recursively try both moves from every stair.

```python
def min_cost_climbing_stairs(cost):
    def go(i):
        if i >= len(cost):
            return 0
        return cost[i] + min(go(i + 1), go(i + 2))
    return min(go(0), go(1))
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** dp[i] = cost to reach stair i = min(dp[i − 1] + cost[i − 1], dp[i − 2] + cost[i − 2]).

```python
def min_cost_climbing_stairs(cost):
    n = len(cost)
    dp = [0] * (n + 1)
    for i in range(2, n + 1):
        dp[i] = min(dp[i - 1] + cost[i - 1], dp[i - 2] + cost[i - 2])
    return dp[n]
```

**Cost:** O(n) time, O(n), or O(1) with two variables space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursively try both moves from every stair. That costs O(2ⁿ) time and O(n) space."

"I can do better: dp[i] = cost to reach stair i = min(dp[i − 1] + cost[i − 1], dp[i − 2] + cost[i − 2]). That's O(n) time and O(n), or O(1) with two variables space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `cost = [1, 100, 1, 1, 1, 100, 1, 1, 100, 1]`

1. dp[2] = min(0 + 100, 0 + 1) = 1
2. dp[3] = min(1 + 1, 0 + 100) = 2
3. dp[4] = min(2 + 1, 1 + 1) = 2
4. dp[5] = min(2 + 1, 2 + 1) = 3
5. dp[6] = min(3 + 100, 2 + 1) = 3
6. dp[7] = min(3 + 1, 3 + 100) = 4
7. dp[8] = min(4 + 1, 3 + 1) = 4
8. dp[9] = min(4 + 100, 4 + 1) = 5
9. dp[10] = min(5 + 1, 4 + 100) = 6

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/min-cost-climbing-stairs)

<a id="q3"></a>
### Q3. House Robber

🟡 Interview standard · *1-D · take or skip*

Adjacent houses can't both be robbed. Return the maximum amount you can rob.

**Examples**

- `nums = [1, 2, 3, 1]` → `4`
- `nums = [2, 7, 9, 3, 1]` → `12`

In [ ]:
def rob(nums):
    """
    Adjacent houses can't both be robbed. Return the maximum amount you can rob.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rob, [
    (([1, 2, 3, 1],), 4),
    (([2, 7, 9, 3, 1],), 12),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursively choose rob-or-skip for every house.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

best(i) = max(best(i − 1), best(i − 2) + nums[i]): skip this house, or rob it plus the best up to two houses back.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recursively choose rob-or-skip for every house.

```python
def rob(nums):
    def go(i):
        if i < 0:
            return 0
        return max(go(i - 1), go(i - 2) + nums[i])
    return go(len(nums) - 1)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** best(i) = max(best(i − 1), best(i − 2) + nums[i]): skip this house, or rob it plus the best up to two houses back.

```python
def rob(nums):
    prev2 = prev1 = 0
    dp = []
    for i, x in enumerate(nums):
        prev2, prev1 = prev1, max(prev1, prev2 + x)
        dp.append(prev1)
    return prev1
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursively choose rob-or-skip for every house. That costs O(2ⁿ) time and O(n) space."

"I can do better: best(i) = max(best(i − 1), best(i − 2) + nums[i]): skip this house, or rob it plus the best up to two houses back. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 3, 1]`

1. House 0: max(skip = 0, rob = 0 + 1) = 1
2. House 1: max(skip = 1, rob = 0 + 2) = 2
3. House 2: max(skip = 2, rob = 1 + 3) = 4
4. House 3: max(skip = 4, rob = 2 + 1) = 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/house-robber)

<a id="q4"></a>
### Q4. House Robber II

🟡 Interview standard · *Circle → two linear runs*

Houses are in a circle, so the first and last are adjacent. Return the maximum you can rob.

**Examples**

- `nums = [2, 3, 2]` → `3`
- `nums = [1, 2, 3, 1]` → `4`
- `nums = [1, 2, 3]` → `3`

In [ ]:
def rob_circle(nums):
    """
    Houses are in a circle, so the first and last are adjacent. Return the
    maximum you can rob.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rob_circle, [
    (([2, 3, 2],), 3),
    (([1, 2, 3, 1],), 4),
    (([1, 2, 3],), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recurse with a flag for whether house 0 was robbed.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

You can't take both ends, so answer = max(rob(nums[1:]), rob(nums[:-1])) using the linear House Robber.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recurse with a flag for whether house 0 was robbed.

```python
def rob_circle(nums):
    n = len(nums)
    if n == 1:
        return nums[0]

    def go(i, first_taken):
        if i >= n:
            return 0
        if i == n - 1 and first_taken:
            return 0
        return max(go(i + 1, first_taken), nums[i] + go(i + 2, first_taken or i == 0))
    return go(0, False)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** You can't take both ends, so answer = max(rob(nums[1:]), rob(nums[:-1])) using the linear House Robber.

```python
def rob_circle(nums):
    if len(nums) == 1:
        return nums[0]

    def line(arr, label):
        a = b = 0
        for x in arr:
            a, b = b, max(b, a + x)
        return b

    return max(line(nums[1:], "first house"), line(nums[:-1], "last house"))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recurse with a flag for whether house 0 was robbed. That costs O(2ⁿ) time and O(n) space."

"I can do better: You can't take both ends, so answer = max(rob(nums[1:]), rob(nums[:-1])) using the linear House Robber. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/house-robber-ii)

<a id="q5"></a>
### Q5. Coin Change

🟡 Interview standard · *Unbounded knapsack · min*

Return the fewest coins needed to make `amount`, or −1 if impossible.

**Examples**

- `coins = [1, 2, 5], amount = 11` → `3`
- `coins = [2], amount = 3` → `-1`
- `coins = [1], amount = 0` → `0`

In [ ]:
def coin_change(coins, amount):
    """
    Return the fewest coins needed to make amount, or −1 if impossible.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(coin_change, [
    (([1, 2, 5], 11), 3),
    (([2], 3), -1),
    (([1], 0), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursively try every coin for every remaining amount.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

dp[a] = fewest coins for amount a = 1 + min(dp[a − c]) over coins c ≤ a; dp[0] = 0.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recursively try every coin for every remaining amount.

```python
def coin_change(coins, amount):
    def go(a):
        if a == 0:
            return 0
        best = float("inf")
        for c in coins:
            if c <= a:
                best = min(best, 1 + go(a - c))
        return best
    r = go(amount)
    return -1 if r == float("inf") else r
```

**Cost:** O(Sⁿ) — exponential time, O(S) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** dp[a] = fewest coins for amount a = 1 + min(dp[a − c]) over coins c ≤ a; dp[0] = 0.

```python
def coin_change(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a] = dp[a - c] + 1
    return -1 if dp[amount] == INF else dp[amount]
```

**Cost:** O(S · n) time, O(S) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursively try every coin for every remaining amount. That costs O(Sⁿ) — exponential time and O(S) space."

"I can do better: dp[a] = fewest coins for amount a = 1 + min(dp[a − c]) over coins c ≤ a; dp[0] = 0. That's O(S · n) time and O(S) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `coins = [1, 3, 4], amount = 6`

1. dp[1] = 1 coins
2. dp[2] = 2 coins
3. dp[3] = 1 coins
4. dp[4] = 1 coins
5. dp[5] = 2 coins
6. dp[6] = 2 coins

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/coin-change)

<a id="q6"></a>
### Q6. Coin Change II

🟡 Interview standard · *Count combinations*

Return the number of combinations of `coins` (unlimited each) that make `amount`.

**Examples**

- `amount = 5, coins = [1, 2, 5]` → `4`
- `amount = 3, coins = [2]` → `0`
- `amount = 10, coins = [10]` → `1`

In [ ]:
def change(amount, coins):
    """
    Return the number of combinations of coins (unlimited each) that make
    amount.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(change, [
    ((5, [1, 2, 5]), 4),
    ((3, [2]), 0),
    ((10, [10]), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recurse on (coin index, remaining amount): use this coin again or move to the next.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

dp[a] = ways to make a. Loop coins on the outside so each combination is counted once in coin order: dp[a] += dp[a − coin].

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recurse on (coin index, remaining amount): use this coin again or move to the next.

```python
def change(amount, coins):
    def go(i, a):
        if a == 0:
            return 1
        if i == len(coins) or a < 0:
            return 0
        return go(i, a - coins[i]) + go(i + 1, a)
    return go(0, amount)
```

**Cost:** O(2^(S/min)) — exponential time, O(S) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** dp[a] = ways to make a. Loop coins on the outside so each combination is counted once in coin order: dp[a] += dp[a − coin].

```python
def change(amount, coins):
    dp = [1] + [0] * amount
    for c in coins:
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]
```

**Cost:** O(S · n) time, O(S) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recurse on (coin index, remaining amount): use this coin again or move to the next. That costs O(2^(S/min)) — exponential time and O(S) space."

"I can do better: dp[a] = ways to make a. Loop coins on the outside so each combination is counted once in coin order: dp[a] += dp[a − coin]. That's O(S · n) time and O(S) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `amount = 5, coins = [1, 2, 5]`

1. After allowing coin 1
2. After allowing coin 2
3. After allowing coin 5

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/coin-change-ii)

<a id="q7"></a>
### Q7. Longest Increasing Subsequence

🟡 Interview standard · *Patience sorting · binary search*

Return the length of the longest strictly increasing subsequence.

**Examples**

- `nums = [10, 9, 2, 5, 3, 7, 101, 18]` → `4`
- `nums = [0, 1, 0, 3, 2, 3]` → `4`
- `nums = [7, 7, 7, 7]` → `1`

In [ ]:
def length_of_lis(nums):
    """
    Return the length of the longest strictly increasing subsequence.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(length_of_lis, [
    (([10, 9, 2, 5, 3, 7, 101, 18],), 4),
    (([0, 1, 0, 3, 2, 3],), 4),
    (([7, 7, 7, 7],), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Classic DP: dp[i] = 1 + max(dp[j]) over j &lt; i with nums[j] &lt; nums[i].

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep tails[k] = smallest possible tail of an increasing subsequence of length k + 1. For each x, binary search the first tail ≥ x and replace it (or append).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Classic DP: dp[i] = 1 + max(dp[j]) over j &lt; i with nums[j] &lt; nums[i].

```python
def length_of_lis(nums):
    dp = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep tails[k] = smallest possible tail of an increasing subsequence of length k + 1. For each x, binary search the first tail ≥ x and replace it (or append).

```python
def length_of_lis(nums):
    from bisect import bisect_left
    tails = []
    for i, x in enumerate(nums):
        k = bisect_left(tails, x)
        if k == len(tails):
            tails.append(x)
        else:
            tails[k] = x
    return len(tails)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Classic DP: dp[i] = 1 + max(dp[j]) over j &lt; i with nums[j] &lt; nums[i]. That costs O(n²) time and O(n) space."

"I can do better: Keep tails[k] = smallest possible tail of an increasing subsequence of length k + 1. For each x, binary search the first tail ≥ x and replace it (or append). That's O(n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [10, 9, 2, 5, 3, 7, 101, 18]`

1. 10 goes to position 0 of tails
2. 9 goes to position 0 of tails
3. 2 goes to position 0 of tails
4. 5 goes to position 1 of tails
5. 3 goes to position 1 of tails
6. 7 goes to position 2 of tails
7. 101 goes to position 3 of tails
8. 18 goes to position 3 of tails

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/longest-increasing-subsequence)

<a id="q8"></a>
### Q8. Longest Common Subsequence

🟡 Interview standard · *Two strings · 2-D table*

Return the length of the longest subsequence common to `text1` and `text2`.

**Examples**

- `a = 'abcde', b = 'ace'` → `3`
- `a = 'abc', b = 'abc'` → `3`
- `a = 'abc', b = 'def'` → `0`

In [ ]:
def longest_common_subsequence(a, b):
    """
    Return the length of the longest subsequence common to text1 and text2.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(longest_common_subsequence, [
    (('abcde', 'ace'), 3),
    (('abc', 'abc'), 3),
    (('abc', 'def'), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recurse on (i, j): match → 1 + f(i+1, j+1), else max(f(i+1, j), f(i, j+1)).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

dp[i][j] = LCS of a[:i] and b[:j]. Equal last letters → dp[i−1][j−1] + 1, else max(dp[i−1][j], dp[i][j−1]).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recurse on (i, j): match → 1 + f(i+1, j+1), else max(f(i+1, j), f(i, j+1)).

```python
def longest_common_subsequence(a, b):
    def go(i, j):
        if i == len(a) or j == len(b):
            return 0
        if a[i] == b[j]:
            return 1 + go(i + 1, j + 1)
        return max(go(i + 1, j), go(i, j + 1))
    return go(0, 0)
```

**Cost:** O(2^(m+n)) time, O(m + n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** dp[i][j] = LCS of a[:i] and b[:j]. Equal last letters → dp[i−1][j−1] + 1, else max(dp[i−1][j], dp[i][j−1]).

```python
def longest_common_subsequence(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]
```

**Cost:** O(m · n) time, O(m · n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recurse on (i, j): match → 1 + f(i+1, j+1), else max(f(i+1, j), f(i, j+1)). That costs O(2^(m+n)) time and O(m + n) space."

"I can do better: dp[i][j] = LCS of a[:i] and b[:j]. Equal last letters → dp[i−1][j−1] + 1, else max(dp[i−1][j], dp[i][j−1]). That's O(m · n) time and O(m · n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `a = 'abcde', b = 'ace'`

1. Row for 'a'
2. Row for 'ab'
3. Row for 'abc'
4. Row for 'abcd'
5. Row for 'abcde'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/longest-common-subsequence)

<a id="q9"></a>
### Q9. Edit Distance

🟡 Interview standard · *Two strings · insert / delete / replace*

Return the minimum number of single-character inserts, deletes and replacements to turn `word1` into `word2`.

**Examples**

- `w1 = 'horse', w2 = 'ros'` → `3`
- `w1 = 'intention', w2 = 'execution'` → `5`

In [ ]:
def min_distance(w1, w2):
    """
    Return the minimum number of single-character inserts, deletes and
    replacements to turn word1 into word2.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_distance, [
    (('horse', 'ros'), 3),
    (('intention', 'execution'), 5),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recurse on (i, j) trying all three operations whenever letters differ.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

dp[i][j] = cost for prefixes. Same letter → dp[i−1][j−1]; else 1 + min(delete dp[i−1][j], insert dp[i][j−1], replace dp[i−1][j−1]).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recurse on (i, j) trying all three operations whenever letters differ.

```python
def min_distance(w1, w2):
    def go(i, j):
        if i == len(w1):
            return len(w2) - j
        if j == len(w2):
            return len(w1) - i
        if w1[i] == w2[j]:
            return go(i + 1, j + 1)
        return 1 + min(go(i + 1, j), go(i, j + 1), go(i + 1, j + 1))
    return go(0, 0)
```

**Cost:** O(3^(m+n)) time, O(m + n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** dp[i][j] = cost for prefixes. Same letter → dp[i−1][j−1]; else 1 + min(delete dp[i−1][j], insert dp[i][j−1], replace dp[i−1][j−1]).

```python
def min_distance(w1, w2):
    m, n = len(w1), len(w2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if w1[i - 1] == w2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[m][n]
```

**Cost:** O(m · n) time, O(m · n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recurse on (i, j) trying all three operations whenever letters differ. That costs O(3^(m+n)) time and O(m + n) space."

"I can do better: dp[i][j] = cost for prefixes. Same letter → dp[i−1][j−1]; else 1 + min(delete dp[i−1][j], insert dp[i][j−1], replace dp[i−1][j−1]). That's O(m · n) time and O(m · n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `w1 = 'horse', w2 = 'ros'`

1. Base cases: turning a prefix into '' (or back) costs its length
2. Row for 'h'
3. Row for 'ho'
4. Row for 'hor'
5. Row for 'hors'
6. Row for 'horse'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/edit-distance)

<a id="q10"></a>
### Q10. Partition Equal Subset Sum

🟡 Interview standard · *0/1 knapsack · reachable sums*

Can `nums` be split into two subsets with equal sums?

**Examples**

- `nums = [1, 5, 11, 5]` → `True`
- `nums = [1, 2, 3, 5]` → `False`

In [ ]:
def can_partition(nums):
    """
    Can nums be split into two subsets with equal sums?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(can_partition, [
    (([1, 5, 11, 5],), True),
    (([1, 2, 3, 5],), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try including or excluding each number (2ⁿ subsets) looking for sum / 2.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

target = sum / 2. dp[s] = can some subset sum to s. For each number loop s downwards so it's used once: dp[s] |= dp[s − x].

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try including or excluding each number (2ⁿ subsets) looking for sum / 2.

```python
def can_partition(nums):
    total = sum(nums)
    if total % 2:
        return False

    def go(i, s):
        if s == 0:
            return True
        if i == len(nums) or s < 0:
            return False
        return go(i + 1, s - nums[i]) or go(i + 1, s)
    return go(0, total // 2)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** target = sum / 2. dp[s] = can some subset sum to s. For each number loop s downwards so it's used once: dp[s] |= dp[s − x].

```python
def can_partition(nums):
    total = sum(nums)
    if total % 2:
        return False
    target = total // 2
    dp = [True] + [False] * target
    for x in nums:
        for s in range(target, x - 1, -1):
            dp[s] = dp[s] or dp[s - x]
    return dp[target]
```

**Cost:** O(n · target) time, O(target) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try including or excluding each number (2ⁿ subsets) looking for sum / 2. That costs O(2ⁿ) time and O(n) space."

"I can do better: target = sum / 2. dp[s] = can some subset sum to s. For each number loop s downwards so it's used once: dp[s] |= dp[s − x]. That's O(n · target) time and O(target) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 5, 11, 5]`

1. After number 1: reachable sums
2. After number 5: reachable sums
3. After number 11: reachable sums
4. After number 5: reachable sums

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/partition-equal-subset-sum)

<a id="q11"></a>
### Q11. Word Break

🟡 Interview standard · *1-D over prefixes*

Can `s` be split into a sequence of words from `wordDict`?

**Examples**

- `s = 'leetcode', word_dict = ['leet', 'code']` → `True`
- `s = 'applepenapple', word_dict = ['apple', 'pen']` → `True`
- `s = 'catsandog', word_dict = ['cats', 'dog', 'sand', 'and', 'cat']` → `False`

In [ ]:
def word_break_dp(s, word_dict):
    """
    Can s be split into a sequence of words from wordDict?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(word_break_dp, [
    (('leetcode', ['leet', 'code']), True),
    (('applepenapple', ['apple', 'pen']), True),
    (('catsandog', ['cats', 'dog', 'sand', 'and', 'cat']), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursively try every dictionary word as the next prefix.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

dp[i] = s[:i] can be split. dp[i] is true if some j &lt; i has dp[j] and s[j:i] in the dictionary.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recursively try every dictionary word as the next prefix.

```python
def word_break_dp(s, word_dict):
    def go(i):
        if i == len(s):
            return True
        return any(s.startswith(w, i) and go(i + len(w)) for w in word_dict)
    return go(0)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** dp[i] = s[:i] can be split. dp[i] is true if some j &lt; i has dp[j] and s[j:i] in the dictionary.

```python
def word_break_dp(s, word_dict):
    words = set(word_dict)
    dp = [True] + [False] * len(s)
    for i in range(1, len(s) + 1):
        for j in range(i):
            if dp[j] and s[j:i] in words:
                dp[i] = True
                break
    return dp[len(s)]
```

**Cost:** O(n² · L) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursively try every dictionary word as the next prefix. That costs O(2ⁿ) time and O(n) space."

"I can do better: dp[i] = s[:i] can be split. dp[i] is true if some j &lt; i has dp[j] and s[j:i] in the dictionary. That's O(n² · L) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'leetcode', word_dict = ['leet', 'code']`

1. dp[1]: can 'l' be split? False
2. dp[2]: can 'le' be split? False
3. dp[3]: can 'lee' be split? False
4. dp[4]: can 'leet' be split? True
5. dp[5]: can 'leetc' be split? False
6. dp[6]: can 'leetco' be split? False
7. dp[7]: can 'leetcod' be split? False
8. dp[8]: can 'leetcode' be split? True

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/word-break)

<a id="q12"></a>
### Q12. Unique Paths

🟡 Interview standard · *Grid · from top and left*

A robot moves only right or down on an m × n grid. Count the paths from top-left to bottom-right.

**Examples**

- `m = 3, n = 7` → `28`
- `m = 3, n = 2` → `3`

In [ ]:
def unique_paths(m, n):
    """
    A robot moves only right or down on an m × n grid. Count the paths from top-
    left to bottom-right.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unique_paths, [
    ((3, 7), 28),
    ((3, 2), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recurse: paths(r, c) = paths(r − 1, c) + paths(r, c − 1).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Fill the grid: the first row and column are 1; every other cell is top + left. One row of memory is enough. (Or C(m+n−2, m−1).)

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recurse: paths(r, c) = paths(r − 1, c) + paths(r, c − 1).

```python
def unique_paths(m, n):
    if m == 1 or n == 1:
        return 1
    return unique_paths(m - 1, n) + unique_paths(m, n - 1)
```

**Cost:** O(2^(m+n)) time, O(m + n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Fill the grid: the first row and column are 1; every other cell is top + left. One row of memory is enough. (Or C(m+n−2, m−1).)

```python
def unique_paths(m, n):
    dp = [[1] * n for _ in range(m)]
    for r in range(1, m):
        for c in range(1, n):
            dp[r][c] = dp[r - 1][c] + dp[r][c - 1]
    return dp[m - 1][n - 1]
```

**Cost:** O(m · n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recurse: paths(r, c) = paths(r − 1, c) + paths(r, c − 1). That costs O(2^(m+n)) time and O(m + n) space."

"I can do better: Fill the grid: the first row and column are 1; every other cell is top + left. One row of memory is enough. (Or C(m+n−2, m−1).) That's O(m · n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `m = 3, n = 4`

1. dp[1][1] = top 1 + left 1 = 2
2. dp[1][2] = top 1 + left 2 = 3
3. dp[1][3] = top 1 + left 3 = 4
4. dp[2][1] = top 2 + left 1 = 3
5. dp[2][2] = top 3 + left 3 = 6
6. dp[2][3] = top 4 + left 6 = 10

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/unique-paths)

<a id="q13"></a>
### Q13. Minimum Path Sum

🟡 Interview standard · *Grid · min of top and left*

Moving only right or down, find the path from top-left to bottom-right with the smallest sum.

**Examples**

- `grid = [[1, 3, 1], [1, 5, 1], [4, 2, 1]]` → `7`
- `grid = [[1, 2, 3], [4, 5, 6]]` → `12`

In [ ]:
def min_path_sum(grid):
    """
    Moving only right or down, find the path from top-left to bottom-right with
    the smallest sum.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_path_sum, [
    (([[1, 3, 1], [1, 5, 1], [4, 2, 1]],), 7),
    (([[1, 2, 3], [4, 5, 6]],), 12),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recurse over both moves from every cell.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

dp[r][c] = grid[r][c] + min(dp[r − 1][c], dp[r][c − 1]); the first row and column only have one way in.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recurse over both moves from every cell.

```python
def min_path_sum(grid):
    m, n = len(grid), len(grid[0])

    def go(r, c):
        if r == m - 1 and c == n - 1:
            return grid[r][c]
        best = float("inf")
        if r + 1 < m:
            best = min(best, go(r + 1, c))
        if c + 1 < n:
            best = min(best, go(r, c + 1))
        return grid[r][c] + best
    return go(0, 0)
```

**Cost:** O(2^(m+n)) time, O(m + n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** dp[r][c] = grid[r][c] + min(dp[r − 1][c], dp[r][c − 1]); the first row and column only have one way in.

```python
def min_path_sum(grid):
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            if r == 0 and c == 0:
                dp[r][c] = grid[0][0]
            elif r == 0:
                dp[r][c] = dp[r][c - 1] + grid[r][c]
            elif c == 0:
                dp[r][c] = dp[r - 1][c] + grid[r][c]
            else:
                dp[r][c] = grid[r][c] + min(dp[r - 1][c], dp[r][c - 1])
    return dp[m - 1][n - 1]
```

**Cost:** O(m · n) time, O(m · n), or O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recurse over both moves from every cell. That costs O(2^(m+n)) time and O(m + n) space."

"I can do better: dp[r][c] = grid[r][c] + min(dp[r − 1][c], dp[r][c − 1]); the first row and column only have one way in. That's O(m · n) time and O(m · n), or O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `grid = [[1, 3, 1], [1, 5, 1], [4, 2, 1]]`

1. dp[0][0] = 1
2. dp[0][1] = 4
3. dp[0][2] = 5
4. dp[1][0] = 2
5. dp[1][1] = 7
6. dp[1][2] = 6
7. dp[2][0] = 6
8. dp[2][1] = 8
9. dp[2][2] = 7

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/minimum-path-sum)

<a id="q14"></a>
### Q14. Decode Ways

🟡 Interview standard · *1-D · one or two digits*

'A' = 1 … 'Z' = 26. Count the ways to decode the digit string `s`.

**Examples**

- `s = '12'` → `2`
- `s = '226'` → `3`
- `s = '06'` → `0`

In [ ]:
def num_decodings(s):
    """
    'A' = 1 … 'Z' = 26. Count the ways to decode the digit string s.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(num_decodings, [
    (('12',), 2),
    (('226',), 3),
    (('06',), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursively take one digit or two digits at every position.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

dp[i] = ways to decode s[:i]. Add dp[i − 1] if s[i − 1] is 1–9, and dp[i − 2] if s[i−2:i] is 10–26.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recursively take one digit or two digits at every position.

```python
def num_decodings(s):
    def go(i):
        if i == len(s):
            return 1
        if s[i] == "0":
            return 0
        ways = go(i + 1)
        if i + 1 < len(s) and 10 <= int(s[i:i + 2]) <= 26:
            ways += go(i + 2)
        return ways
    return go(0)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** dp[i] = ways to decode s[:i]. Add dp[i − 1] if s[i − 1] is 1–9, and dp[i − 2] if s[i−2:i] is 10–26.

```python
def num_decodings(s):
    n = len(s)
    dp = [1] + [0] * n
    for i in range(1, n + 1):
        if s[i - 1] != "0":
            dp[i] += dp[i - 1]
        if i >= 2 and 10 <= int(s[i - 2:i]) <= 26:
            dp[i] += dp[i - 2]
    return dp[n]
```

**Cost:** O(n) time, O(n), or O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursively take one digit or two digits at every position. That costs O(2ⁿ) time and O(n) space."

"I can do better: dp[i] = ways to decode s[:i]. Add dp[i − 1] if s[i − 1] is 1–9, and dp[i − 2] if s[i−2:i] is 10–26. That's O(n) time and O(n), or O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = '11106'`

1. dp[1] for '1' = 1
2. dp[2] for '11' = 2
3. dp[3] for '111' = 3
4. dp[4] for '1110' = 2
5. dp[5] for '11106' = 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/decode-ways)

<a id="q15"></a>
### Q15. Longest Palindromic Substring

🟡 Interview standard · *Expand around centres*

Return the longest palindromic substring of `s`.

**Examples**

- `s = 'babad'` → `'bab'`
- `s = 'cbbd'` → `'bb'`
- `s = 'a'` → `'a'`

In [ ]:
def longest_palindrome(s):
    """
    Return the longest palindromic substring of s.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(longest_palindrome, [(("babad",), "bab"), (("cbbd",), "bb"), (("a",), "a")], verify=lambda a, out: out == out[::-1] and out in a[0] and len(out) == max(len(a[0][i:j]) for i in range(len(a[0])) for j in range(i + 1, len(a[0]) + 1) if a[0][i:j] == a[0][i:j][::-1]))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every substring with a reverse comparison.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Every palindrome has a centre — a character or a gap. Expand outward from each of the 2n − 1 centres while the ends match. (The DP table version is O(n²) space.)

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Check every substring with a reverse comparison.

```python
def longest_palindrome(s):
    best = ""
    for i in range(len(s)):
        for j in range(i, len(s)):
            t = s[i:j + 1]
            if t == t[::-1] and len(t) > len(best):
                best = t
    return best
```

**Cost:** O(n³) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Every palindrome has a centre — a character or a gap. Expand outward from each of the 2n − 1 centres while the ends match. (The DP table version is O(n²) space.)

```python
def longest_palindrome(s):
    lo = hi = 0
    for c in range(2 * len(s) - 1):
        l, r = c // 2, (c + 1) // 2
        while l >= 0 and r < len(s) and s[l] == s[r]:
            l, r = l - 1, r + 1
        if r - l - 1 > hi - lo + 1:
            lo, hi = l + 1, r - 1
    return s[lo:hi + 1]
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every substring with a reverse comparison. That costs O(n³) time and O(1) space."

"I can do better: Every palindrome has a centre — a character or a gap. Expand outward from each of the 2n − 1 centres while the ends match. (The DP table version is O(n²) space.) That's O(n²) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'cabbad'`

1. Centre at 0: expands to 'c'
2. Centre between 0 and 1: expands to ''
3. Centre at 1: expands to 'a'
4. Centre between 1 and 2: expands to ''
5. Centre at 2: expands to 'b'
6. Centre between 2 and 3: expands to 'abba'
7. Centre at 3: expands to 'b'
8. Centre between 3 and 4: expands to ''
9. Centre at 4: expands to 'a'
10. Centre between 4 and 5: expands to ''
11. Centre at 5: expands to 'd'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/longest-palindromic-substring)

<a id="q16"></a>
### Q16. Maximum Product Subarray

🟡 Interview standard · *Track max and min*

Return the largest product of a contiguous subarray.

**Examples**

- `nums = [2, 3, -2, 4]` → `6`
- `nums = [-2, 0, -1]` → `0`
- `nums = [-2, 3, -4]` → `24`

In [ ]:
def max_product(nums):
    """
    Return the largest product of a contiguous subarray.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_product, [
    (([2, 3, -2, 4],), 6),
    (([-2, 0, -1],), 0),
    (([-2, 3, -4],), 24),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every subarray with a running product.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A negative number turns the smallest product into the largest. Keep both the max and min product ending at i: candidates are x, x · max, x · min.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every subarray with a running product.

```python
def max_product(nums):
    best = nums[0]
    for i in range(len(nums)):
        p = 1
        for j in range(i, len(nums)):
            p *= nums[j]
            best = max(best, p)
    return best
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** A negative number turns the smallest product into the largest. Keep both the max and min product ending at i: candidates are x, x · max, x · min.

```python
def max_product(nums):
    hi = lo = best = nums[0]
    for i in range(1, len(nums)):
        x = nums[i]
        hi, lo = max(x, x * hi, x * lo), min(x, x * hi, x * lo)
        best = max(best, hi)
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every subarray with a running product. That costs O(n²) time and O(1) space."

"I can do better: A negative number turns the smallest product into the largest. Keep both the max and min product ending at i: candidates are x, x · max, x · min. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [2, -5, -2, -4, 3]`

1. x = -5: max ending here -5, min ending here -10
2. x = -2: max ending here 20, min ending here -2
3. x = -4: max ending here 8, min ending here -80
4. x = 3: max ending here 24, min ending here -240

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/maximum-product-subarray)

<a id="q17"></a>
### Q17. Target Sum

🟡 Interview standard · *Subset-sum count*

Put + or − before every number in `nums`. Count the expressions that equal `target`.

**Examples**

- `nums = [1, 1, 1, 1, 1], target = 3` → `5`
- `nums = [1], target = 1` → `1`

In [ ]:
def find_target_sum_ways(nums, target):
    """
    Put + or − before every number in nums. Count the expressions that equal
    target.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_target_sum_ways, [
    (([1, 1, 1, 1, 1], 3), 5),
    (([1], 1), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try both signs for every number.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Let P be the numbers with +. Then P − (total − P) = target, so sum(P) = (total + target) / 2. Count subsets with that sum (0/1 knapsack counting, loop sums downward).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try both signs for every number.

```python
def find_target_sum_ways(nums, target):
    def go(i, s):
        if i == len(nums):
            return int(s == target)
        return go(i + 1, s + nums[i]) + go(i + 1, s - nums[i])
    return go(0, 0)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Let P be the numbers with +. Then P − (total − P) = target, so sum(P) = (total + target) / 2. Count subsets with that sum (0/1 knapsack counting, loop sums downward).

```python
def find_target_sum_ways(nums, target):
    total = sum(nums)
    if abs(target) > total or (total + target) % 2:
        return 0
    goal = (total + target) // 2
    dp = [1] + [0] * goal
    for x in nums:
        for s in range(goal, x - 1, -1):
            dp[s] += dp[s - x]
    return dp[goal]
```

**Cost:** O(n · S) time, O(S) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try both signs for every number. That costs O(2ⁿ) time and O(n) space."

"I can do better: Let P be the numbers with +. Then P − (total − P) = target, so sum(P) = (total + target) / 2. Count subsets with that sum (0/1 knapsack counting, loop sums downward). That's O(n · S) time and O(S) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 1, 3], target = 1`

1. After 1: ways to pick a '+' group summing to each value (need 4)
2. After 2: ways to pick a '+' group summing to each value (need 4)
3. After 1: ways to pick a '+' group summing to each value (need 4)
4. After 3: ways to pick a '+' group summing to each value (need 4)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/target-sum)

<a id="q18"></a>
### Q18. Best Time to Buy and Sell Stock with Cooldown

🟡 Interview standard · *State machine DP*

Unlimited trades, but after selling you must wait one day before buying again. Maximise profit.

**Examples**

- `prices = [1, 2, 3, 0, 2]` → `3`
- `prices = [1]` → `0`

In [ ]:
def max_profit_cooldown(prices):
    """
    Unlimited trades, but after selling you must wait one day before buying
    again. Maximise profit.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_profit_cooldown, [
    (([1, 2, 3, 0, 2],), 3),
    (([1],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recurse on (day, holding?) trying buy / sell / wait.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Three states per day: hold (own a stock), sold (sold today), rest (free to buy). hold = max(hold, rest − p); sold = hold + p; rest = max(rest, sold).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recurse on (day, holding?) trying buy / sell / wait.

```python
def max_profit_cooldown(prices):
    def go(i, holding):
        if i >= len(prices):
            return 0
        wait = go(i + 1, holding)
        if holding:
            return max(wait, prices[i] + go(i + 2, False))
        return max(wait, -prices[i] + go(i + 1, True))
    return go(0, False)
```

**Cost:** O(2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Three states per day: hold (own a stock), sold (sold today), rest (free to buy). hold = max(hold, rest − p); sold = hold + p; rest = max(rest, sold).

```python
def max_profit_cooldown(prices):
    hold, sold, rest = float("-inf"), 0, 0
    for i, p in enumerate(prices):
        hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
    return max(sold, rest)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recurse on (day, holding?) trying buy / sell / wait. That costs O(2ⁿ) time and O(n) space."

"I can do better: Three states per day: hold (own a stock), sold (sold today), rest (free to buy). hold = max(hold, rest − p); sold = hold + p; rest = max(rest, sold). That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `prices = [1, 2, 3, 0, 2]`

1. Day 0, price 1
2. Day 1, price 2
3. Day 2, price 3
4. Day 3, price 0
5. Day 4, price 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/best-time-to-buy-and-sell-stock-with-cooldown)

<a id="q19"></a>
### Q19. Maximal Square

🟡 Interview standard · *Grid · min of three neighbours*

In a binary matrix of '0' and '1', return the area of the largest square of 1s.

**Examples**

- `matrix = [['1', '0', '1', '0', '0'], ['1', '0', '1', '1', '1'], ['1', '1', '1', '1', '1'], ['1', '0', '0', '1', '0']]` → `4`
- `matrix = [['0', '1'], ['1', '0']]` → `1`
- `matrix = [['0']]` → `0`

In [ ]:
def maximal_square(matrix):
    """
    In a binary matrix of '0' and '1', return the area of the largest square of
    1s.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(maximal_square, [
    (([['1', '0', '1', '0', '0'], ['1', '0', '1', '1', '1'], ['1', '1', '1', '1', '1'], ['1', '0', '0', '1', '0']],), 4),
    (([['0', '1'], ['1', '0']],), 1),
    (([['0']],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every cell as a top-left corner, grow the square while it stays all 1s.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

dp[r][c] = side of the largest square whose bottom-right corner is (r, c) = 1 + min(top, left, top-left) when the cell is '1'.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every cell as a top-left corner, grow the square while it stays all 1s.

```python
def maximal_square(matrix):
    m, n, best = len(matrix), len(matrix[0]), 0
    for r in range(m):
        for c in range(n):
            k = 0
            while r + k < m and c + k < n and all(matrix[r + i][c + j] == "1" for i in range(k + 1) for j in range(k + 1)):
                k += 1
            best = max(best, k)
    return best * best
```

**Cost:** O((m·n)²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** dp[r][c] = side of the largest square whose bottom-right corner is (r, c) = 1 + min(top, left, top-left) when the cell is '1'.

```python
def maximal_square(matrix):
    m, n = len(matrix), len(matrix[0])
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for r in range(1, m + 1):
        for c in range(1, n + 1):
            if matrix[r - 1][c - 1] == "1":
                dp[r][c] = 1 + min(dp[r - 1][c], dp[r][c - 1], dp[r - 1][c - 1])
                best = max(best, dp[r][c])
    return best * best
```

**Cost:** O(m · n) time, O(m · n), or O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every cell as a top-left corner, grow the square while it stays all 1s. That costs O((m·n)²) time and O(1) space."

"I can do better: dp[r][c] = side of the largest square whose bottom-right corner is (r, c) = 1 + min(top, left, top-left) when the cell is '1'. That's O(m · n) time and O(m · n), or O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `matrix = [['1', '0', '1', '0', '0'], ['1', '0', '1', '1', '1'], ['1', '1', '1', '1', '1'], ['1', '0', '0', '1', '0']]`

1. Row 0 done; largest side so far 1
2. Row 1 done; largest side so far 1
3. Row 2 done; largest side so far 2
4. Row 3 done; largest side so far 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/maximal-square)

<a id="q20"></a>
### Q20. Burst Balloons

🔴 Stretch · *Interval DP · last balloon*

Bursting balloon i earns nums[left] · nums[i] · nums[right] with its current neighbours (outside = 1). Maximise the total.

**Examples**

- `nums = [3, 1, 5, 8]` → `167`
- `nums = [1, 5]` → `10`

In [ ]:
def max_coins(nums):
    """
    Bursting balloon i earns nums[left] · nums[i] · nums[right] with its current
    neighbours (outside = 1). Maximise the total.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_coins, [
    (([3, 1, 5, 8],), 167),
    (([1, 5],), 10),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every order of bursting balloons.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Think of the LAST balloon k burst inside (l, r): its neighbours are then l and r. dp[l][r] = max over k of dp[l][k] + a[l]·a[k]·a[r] + dp[k][r], filled by increasing interval length.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every order of bursting balloons.

```python
def max_coins(nums):
    def go(arr):
        if not arr:
            return 0
        best = 0
        for i in range(len(arr)):
            left = arr[i - 1] if i > 0 else 1
            right = arr[i + 1] if i + 1 < len(arr) else 1
            best = max(best, left * arr[i] * right + go(arr[:i] + arr[i + 1:]))
        return best
    return go(list(nums))
```

**Cost:** O(n!) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Think of the LAST balloon k burst inside (l, r): its neighbours are then l and r. dp[l][r] = max over k of dp[l][k] + a[l]·a[k]·a[r] + dp[k][r], filled by increasing interval length.

```python
def max_coins(nums):
    a = [1] + nums + [1]
    n = len(a)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n):
        for l in range(n - length):
            r = l + length
            for k in range(l + 1, r):
                dp[l][r] = max(dp[l][r], dp[l][k] + a[l] * a[k] * a[r] + dp[k][r])
    return dp[0][n - 1]
```

**Cost:** O(n³) time, O(n²) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every order of bursting balloons. That costs O(n!) time and O(n) space."

"I can do better: Think of the LAST balloon k burst inside (l, r): its neighbours are then l and r. dp[l][r] = max over k of dp[l][k] + a[l]·a[k]·a[r] + dp[k][r], filled by increasing interval length. That's O(n³) time and O(n²) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [3, 1, 5]`

1. Intervals of length 2 done
2. Intervals of length 3 done
3. Intervals of length 4 done

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/dp/burst-balloons)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What two properties must a problem have for DP?</b></summary>

**Optimal substructure** (optimal solutions are built from optimal sub-solutions) and **overlapping sub-problems** (the same sub-problems recur). Without overlap, use divide and conquer; with a safe local choice, greedy may suffice.

</details>

<details>
<summary><b>2. Memoisation vs tabulation?</b></summary>

Memoisation is top-down: write the recursion and cache results; computes only needed states but has recursion overhead. Tabulation is bottom-up: fill a table in dependency order; no recursion and easy to space-optimise.

</details>

<details>
<summary><b>3. How do you find the time complexity of a DP solution?</b></summary>

Number of distinct states × time per state transition. For example, coin change has (amount + 1) states, each trying n coins → O(amount × n).

</details>

<details>
<summary><b>4. What is the difference between DP and greedy?</b></summary>

Greedy makes one locally optimal choice per step and never reconsiders; it's only correct when a greedy-choice property can be proven. DP considers all choices for each sub-problem and keeps the best, so it's correct whenever optimal substructure holds — at a higher cost.

</details>

<details>
<summary><b>5. Why does 0/1 knapsack iterate capacity backwards in a 1-D array?</b></summary>

Going from high to low capacity means `dp[c − w]` still holds the previous item's row, so each item is counted at most once. Going forwards would let the same item be reused (that's unbounded knapsack).

</details>

<details>
<summary><b>6. Why is knapsack called pseudo-polynomial?</b></summary>

O(n · W) is polynomial in the numeric value W, but W can be exponential in the number of bits used to write it. Knapsack is NP-hard in general.

</details>

<details>
<summary><b>7. How do you reconstruct the actual solution, not just its value?</b></summary>

Store the choice made at each state (or recompute it by checking which transition produced the stored value), then walk back from the answer state to the base case.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why plain recursion is slow
- [ ] I can explain: Two ways to do DP
- [ ] I can explain: 1-D DP: take it or skip it
- [ ] I can explain: Knapsack: choosing items under a limit
- [ ] I can explain: Grid DP
- [ ] I can explain: Two-string DP
- [ ] I can explain: Harder DP shapes

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Maximum Product Subarray
- [ ] [Q17](#q17) Target Sum
- [ ] [Q18](#q18) Best Time to Buy and Sell Stock with Cooldown
- [ ] [Q19](#q19) Maximal Square
- [ ] [Q20](#q20) Burst Balloons

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.